# Day 2. Building the box

One call sits at the center of this whole week: **`create_agent`**. We start
with it bare, then grow it one argument at a time: **messages → `system_prompt`
→ `tools` → `response_format`**, and close with the differentiation that
decides every build: **`create_agent` vs `with_structured_output`**.

How to use this notebook: run top to bottom. Cells marked **Your turn** are
yours, we compare results in the room.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · The smallest agent that exists

Two arguments, a model and its instructions, and it is already an agent.
No tools yet, so it can talk and nothing else: the locked room, addressable.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=llm,
    system_prompt="You are the GIU booking assistant. Be brief.",
)

result = agent.invoke({"messages": [
    HumanMessage("I need a room in Berlin next week.")
]})
print(result["messages"][-1].content)

### The map for the whole week

```python
create_agent(
    model=...,            # the brain
    system_prompt=...,    # today · the behavior dial
    tools=[...],          # today · give it hands
    response_format=...,  # today · structured output
    middleware=[...],     # tomorrow · skills, guardrails
    checkpointer=...,     # tomorrow · memory that survives
)
```

Every section below adds one argument. Nothing else about the call ever
changes.

## 2 · Everything is messages

`agent.invoke` eats `{"messages": [...]}` and returns the grown transcript in
`result["messages"]`. Four roles: `system`, `user (human)`, `assistant (ai)`,
`tool`. The transcript IS the context window from yesterday.

In [ ]:
for m in result["messages"]:
    print(m.type.upper().ljust(6), "→", str(m.content)[:100])

In [ ]:
# continue a conversation: pass the OLD transcript back, grown by one turn
msgs = result["messages"] + [HumanMessage("Actually, make it two nights.")]
result2 = agent.invoke({"messages": msgs})
print(result2["messages"][-1].content)

### The agent remembers nothing, the list does

A fresh invoke has no past. "Memory" is the harness re-sending the list
(tomorrow: `checkpointer` does it for you).

In [ ]:
fresh = agent.invoke({"messages": [
    HumanMessage("How many nights did I want?")]})
print(fresh.get("messages")[-1].content)

In [ ]:
# history surgery: drop the correction and watch the answer change
surgery = [m for m in result2["messages"][:-1] if "two nights" not in str(m.content)]
out = agent.invoke({"messages": surgery + [HumanMessage("How many nights did I want?")]})
print(out["messages"][-1].content)

**Your turn**, tell the agent your name. Then ask *"What's my name?"* twice:
once **continuing the transcript**, once in a **fresh invoke**. One works, one
doesn't, say why in one sentence.

In [ ]:
# your code here


### `system_prompt` is the behavior dial

Same model, same question, three `create_agent` calls.

In [ ]:
QUESTION = HumanMessage("Can I get a room in Berlin?")

PROMPTS = {
    "clerk": "You are a precise booking clerk. Collect city, dates and budget before acting. No small talk.",
    "guide": "You are a friendly campus guide. Suggest neighbourhoods and tips; you can never book anything.",
    "robot": "Reply ONLY with valid JSON: {\"intent\": ..., \"missing_info\": [...]}. No free text.",
}

for name, sp in PROMPTS.items():
    a = create_agent(model=llm, system_prompt=sp)
    out = a.invoke({"messages": [QUESTION]})
    print(f"── {name} " + "─" * 40)
    print(out["messages"][-1].content[:300], "\n")

**Your turn**, write a system prompt that makes the agent politely refuse all
booking requests and redirect to the official university travel portal,
without ever mentioning it was told to do so.

In [ ]:
# your system prompt here


## 3 · Add a tool: `tools=[...]`

A tool is a function the model can *read*: name, docstring, typed arguments.
It never executes anything, it writes a **request**, the harness runs it.

In [ ]:
from langchain.tools import tool

HOTELS = {
    "Hotel Anders":  {"free": True,  "eur": 92},
    "Pension Krumm": {"free": True,  "eur": 61},
    "Grand Mitte":   {"free": False, "eur": 210},
}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

INSTRUCTIONS = """You are the GIU booking assistant.
- Always check availability before recommending.
- Prefer the cheapest available hotel unless told otherwise.
- Report prices in EUR. Never invent a hotel or a price."""

agent = create_agent(model=llm,
    system_prompt=INSTRUCTIONS,
    tools=[check_availability, list_hotels])   # ← one new argument

result = agent.invoke({"messages": [
    HumanMessage("Find me a room in Berlin, cheap but decent.")]})
print(result["messages"][-1].content)

### Read the trace. Always.

The loop just ran for real. *"What did it have in context?"* is now a `for`
loop, and this is where every agent bug is found.

In [ ]:
for m in result["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:110])

### Under the hood: the round trip, once by hand

You do this exactly once, so `create_agent` is never magic to you again.

In [ ]:
import json
from langchain.messages import SystemMessage, ToolMessage

llm_tools = llm.bind_tools([check_availability, list_hotels])
TOOLS = {"check_availability": check_availability, "list_hotels": list_hotels}

msgs = [SystemMessage(INSTRUCTIONS),
        HumanMessage("What's the cheapest available hotel?")]

reply = llm_tools.invoke(msgs)
print("the model proposes:", reply.tool_calls)

while reply.tool_calls:                       # the loop's body, by hand
    msgs.append(reply)
    for call in reply.tool_calls:
        out = TOOLS[call["name"]].invoke(call["args"])
        msgs.append(ToolMessage(json.dumps(out), tool_call_id=call["id"]))
    reply = llm_tools.invoke(msgs)

print(reply.content)

**Your turn**, add a `book_room(hotel: str, nights: int) -> dict` tool that
returns a fake confirmation code, give it to the AGENT (not the manual loop),
ask *"Book me the cheapest room for 2 nights"*, and read the two-step plan in
the trace.

In [ ]:
# your tool here


## 4 · Add structure: `response_format`

Free text is for people. With `response_format`, the agent **works with its
tools first**, then delivers its final answer in your schema, no parsing, no
extra call. (No tools and a single step? That is the *direct form*, section 6.)

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class BookingPlan(BaseModel):
    """The final plan after checking real availability."""
    hotel: str
    eur_per_night: int
    status: Literal["available", "unavailable", "needs_human"]
    note: str = Field(description="One line for the user")

planner = create_agent(model=llm,
    system_prompt=INSTRUCTIONS,
    tools=[check_availability, list_hotels],
    response_format=ToolStrategy(BookingPlan))   # ← one new argument

result = planner.invoke({"messages": [HumanMessage(
    "Cheapest room in Berlin for 2 nights?")]})
plan = result["structured_response"]
plan

In [ ]:
# it's DATA now, your code can branch on it. And read the trace:
# the agent CHECKED real availability before reporting in your schema.
for m in result["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:100])

Notes on the schema, it is a **prompt, not a comment**: the docstring and
every `Field(description=...)` are read by the model; constraints reject junk
and trigger a retry; `None` and `Literal` keep the model honest.

Strategies: `ToolStrategy` works on any tool-calling model; `ProviderStrategy`
uses native structured output; pass a raw schema and it auto-picks.

**Your turn**, extend `BookingPlan` with
`alternatives: list[str] = Field(description="Up to 2 backup hotels")` and
watch the trace: does the agent check more hotels now?

In [ ]:
# your extended schema here


## 5 · Same call, different agents

Only the arguments change. Ever. Two more agents in a few lines:

In [ ]:
CALENDAR = {"Mon": ["10-12"], "Tue": [], "Wed": ["14-18"], "Thu": ["9-11", "15-17"], "Fri": []}
CATALOG = [
    {"code": "CS402", "title": "Distributed Systems", "slot": "Tue 10-12"},
    {"code": "AI310", "title": "Applied ML",          "slot": "Thu 15-17"},
    {"code": "SE250", "title": "Software Architecture","slot": "Fri 10-12"},
]

@tool
def free_slots(day: str) -> list:
    """Booked time ranges for a weekday (empty list = fully free)."""
    return CALENDAR.get(day, ["unknown day"])

@tool
def catalog_search(topic: str) -> list:
    """Courses whose title mentions the topic."""
    return [c for c in CATALOG if topic.lower() in c["title"].lower()]

study_planner = create_agent(model=llm,
    system_prompt="Plan the student's week. Never double-book. Show trade-offs briefly.",
    tools=[free_slots, catalog_search])

out = study_planner.invoke({"messages": [HumanMessage(
    "I want to take something about ML, does it clash with my week?")]})
print(out["messages"][-1].content)

In [ ]:
# email triage: a lookup tool + a structured verdict
STUDENTS = {"mona": {"status": "active", "flags": 0},
            "youssef": {"status": "locked", "flags": 2}}

@tool
def lookup_student(name: str) -> dict:
    """Account status and security flags for a student."""
    return STUDENTS.get(name.lower(), {"error": "unknown student"})

class TriageDecision(BaseModel):
    """Routing decision for one incoming email."""
    category: Literal["question", "complaint", "request", "spam"]
    urgency: int = Field(ge=1, le=5)
    route_to: Literal["auto_reply", "advisor", "security"]
    draft_reply: str = Field(description="A reply draft. Empty if route_to != auto_reply.")

triage = create_agent(model=llm,
    system_prompt="Triage student emails. Look the student up first; anything mentioning account access or suspicious links routes to security.",
    tools=[lookup_student],
    response_format=ToolStrategy(TriageDecision))

out = triage.invoke({"messages": [HumanMessage(
    "This is Youssef, someone sent me a link to 'verify my student account' and now I can't log in!!")]})
out["structured_response"]

**Your turn**, a data agent: embed a small grades dataset (a list of dicts),
give it one `csv_query`-style tool and the instruction *"show your working;
never invent a number."* Ask it for the average grade per course.

In [ ]:
# your data agent here


## 6 · The direct form: `with_structured_output`

Most steps don't need a loop. They need **one typed call**, text in, validated
data out. No tools, no planning: it cannot act, only transform. This is the
workhorse of workflows.

In [ ]:
from datetime import date

class BookingRequest(BaseModel):
    """One hotel booking ask, extracted from one message."""
    city: str = Field(description="Destination city")
    check_in: date
    nights: int = Field(ge=1, le=30)
    budget_per_night: int | None = Field(
        None, description="EUR per night. None if not mentioned.")

extractor = llm.with_structured_output(BookingRequest)   # ← the direct form

email = """Hi! Flying in for the AI Connects workshop, need somewhere to stay
in Berlin from July 27, probably 3 nights. Nothing fancy."""

req = extractor.invoke(email)
print(req)
print("budget given?", req.budget_per_night is not None)   # honest None

### A workflow chains direct calls with your code

In [ ]:
def booking_workflow(email_text: str) -> str:
    req = extractor.invoke(email_text)                   # 1 · the direct call

    if req.nights > 7:                                    # 2 · YOUR rule
        return f"ESCALATE: {req.nights} nights needs human approval."

    cheapest = min((h for h in HOTELS if HOTELS[h]["free"]),
                   key=lambda h: HOTELS[h]["eur"])        # 3 · plain code
    quote = check_availability.invoke({"hotel": cheapest})

    draft = llm.invoke(                                   # 4 · draft
        f"Write a 2-sentence confirmation for {req.nights} nights at "
        f"{cheapest} ({quote['eur']} EUR/night), arriving {req.check_in}.")
    return draft.content

print(booking_workflow(email))

### The differentiation

| | `with_structured_output` | `create_agent` |
|---|---|---|
| Shape | one typed call | a loop that plans |
| Tools | none, it cannot act | yes. It looks up and acts |
| Control flow | **yours**: order, branches, retries | the model's, until done |
| Typed output | always | add `response_format` |
| Cost | one call | as many as the loop takes |
| Reach for it | **first, by default** | when the path can't be drawn |

> **The rule: use the least powerful abstraction that does the job.
> Start with the direct call; upgrade to the agent when the path needs
> planning.**

**Your turn**, a `CourseFeedback` schema (`course_code`, `sentiment`,
`topics`, `needs_followup`) with the **direct form**, run on a paragraph of
invented feedback. Then: for your take-home task, which form, and why? One
paragraph, as a comment.

In [ ]:
# CourseFeedback with the direct form + your argument here


---
## Wrap

One call, grown all day: `create_agent(model, system_prompt)` → messages and
the proof it remembers nothing → `+ tools` and the visible loop → `+
response_format` and answers as data → and the differentiation:
**`with_structured_output` for typed steps you control, `create_agent` for
paths that need planning.**

**Tomorrow:** the two arguments left on the map, `middleware` (skills you'll
build the injector for, and guardrails) and `checkpointer` (memory that
survives).